In [ ]:
%pip install -q -U pandas openai pillow tqdm

In [ ]:
import base64, io, json, os, re, shutil, time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
import pandas as pd
from PIL import Image, ImageOps
from openai import OpenAI
from tqdm.auto import tqdm

In [ ]:
client = OpenAI(base_url="https://integrate.api.nvidia.com/v1", api_key="YOUR_API_KEY_HERE")

MODEL = "google/gemma-4-31b-it"
USE_SCREENSHOT = True
MAX_SIDE = 1536
WORKERS = 4
RETRIES = 3
LIMIT = None

try: PROJECT_DIR = Path(__file__).resolve().parents[1]
except NameError: PROJECT_DIR = Path.cwd().parent
REPO = PROJECT_DIR / "metadata/repository.csv"
BACKUP = REPO.with_name("repository.bak.csv")
IMAGES_DIR = PROJECT_DIR / "data/processed/images"
SCREENSHOTS_DIR = PROJECT_DIR / "data/processed/screenshots"

In [ ]:
if not BACKUP.exists(): shutil.copy(REPO, BACKUP)

df = pd.read_csv(REPO)
if "image_descriptions" not in df.columns:
    df.insert(df.columns.get_loc("images") + 1, "image_descriptions", None)
df["image_descriptions"] = df["image_descriptions"].astype("object")
df.loc[df["image_descriptions"].isna() & (df["has_image"] != 1), "image_descriptions"] = "{}"

def save():
    tmp = REPO.with_suffix(".tmp")
    df.to_csv(tmp, index=False)
    tmp.replace(REPO)

save()

pending = df[(df["has_image"] == 1) & df["image_descriptions"].isna()].drop_duplicates("screenshot")
questions = [
    {"screenshot": r.screenshot, "content": r.content, "options": json.loads(r.options), "images": json.loads(r.images)}
    for r in pending.itertuples()
]

def needed_files(q):
    files = [IMAGES_DIR / n for n in q["images"]]
    return files + [SCREENSHOTS_DIR / q["screenshot"]] if USE_SCREENSHOT else files

missing = {q["screenshot"]: [p.name for p in needed_files(q) if not p.exists()] for q in questions}
missing = {k: v for k, v in missing.items() if v}
questions = [q for q in questions if q["screenshot"] not in missing]
print(f"{len(questions)} questions to describe ({sum(len(q['images']) for q in questions)} images) | {len(missing)} with missing files")
if missing: print(list(missing.items())[:5])

In [ ]:
PROMPT = """Descreva cada imagem desta questão da Olimpíada Brasileira de Robótica para que um modelo que só lê texto consiga resolvê-la.

Enunciado:
{content}

Alternativas:
{options}

Imagens a descrever: {names}
{note}
Regras:
- Descreva só o que é visível, de forma objetiva. Não resolva a questão nem indique alternativas.
- Transcreva textos, números, rótulos e legendas exatamente como aparecem.
- Informe posições da perspectiva de quem olha e cores que tenham significado. Em grades, mapas e tabelas, dê linhas, colunas, coordenadas e contagens relevantes.
- Cada descrição deve ser independente. Não repita o enunciado nem as alternativas.
- Português do Brasil, texto corrido sem markdown: curto para imagens simples, detalhado para esquemas, grades e mapas.

Responda só com JSON: {{"descriptions": [{{"image": "arquivo.png", "description": "..."}}]}}"""

NOTE = "A primeira imagem enviada é a captura da página, só como contexto de layout: NÃO a descreva.\n"

def encode_image(path):
    img = ImageOps.exif_transpose(Image.open(path))
    if img.mode in ("RGBA", "LA", "P"):
        img = img.convert("RGBA")
        bg = Image.new("RGB", img.size, "white")
        bg.paste(img, mask=img.split()[-1])
        img = bg
    else:
        img = img.convert("RGB")
    img.thumbnail((MAX_SIDE, MAX_SIDE))
    buf = io.BytesIO()
    img.save(buf, "JPEG", quality=90)
    return "data:image/jpeg;base64," + base64.b64encode(buf.getvalue()).decode()

def build_messages(q):
    text = PROMPT.format(
        content=q["content"].strip(),
        options="\n".join(f"{k}) {v}" for k, v in q["options"].items()),
        names=", ".join(f"`{n}`" for n in q["images"]),
        note=NOTE if USE_SCREENSHOT else "",
    )
    parts = [{"type": "text", "text": text}]
    if USE_SCREENSHOT:
        parts.append({"type": "image_url", "image_url": {"url": encode_image(SCREENSHOTS_DIR / q["screenshot"])}})
    for n in q["images"]:
        parts.append({"type": "text", "text": f"Imagem `{n}`:"})
        parts.append({"type": "image_url", "image_url": {"url": encode_image(IMAGES_DIR / n)}})
    return [{"role": "user", "content": parts}]

def parse(text):
    data = json.loads(re.search(r"\{.*\}", text, re.S).group())
    return {d["image"].strip("` \n"): d["description"].strip() for d in data["descriptions"]}

def describe(q):
    messages, last = build_messages(q), None
    for attempt in range(RETRIES):
        try:
            r = client.chat.completions.create(model=MODEL, messages=build_messages(q), temperature=0.2, max_tokens=16384)
            out = parse(r.choices[0].message.content)
            if set(out) != set(q["images"]) or not all(out.values()):
                raise ValueError(f"returned images {sorted(out)} != expected {q['images']}")
            return {n: out[n] for n in q["images"]}
        except Exception as e:
            last = e
            time.sleep(2 ** attempt)
    raise last

In [ ]:
todo = [q for q in questions if df.loc[df["screenshot"] == q["screenshot"], "image_descriptions"].isna().all()]
if LIMIT: todo = todo[:LIMIT]
failed = {}

with ThreadPoolExecutor(WORKERS) as ex:
    futs = {ex.submit(describe, q): q for q in todo}
    for f in tqdm(as_completed(futs), total=len(futs)):
        q = futs[f]
        try:
            df.loc[df["screenshot"] == q["screenshot"], "image_descriptions"] = json.dumps(f.result(), ensure_ascii=False)
            save()
        except Exception as e:
            failed[q["screenshot"]] = str(e)
            print(q["screenshot"], e)

left = df[(df["has_image"] == 1) & df["image_descriptions"].isna()]
print(f"failures: {len(failed)} | rows with images but no description: {len(left)} of {int(df['has_image'].sum())}")